<a href="https://colab.research.google.com/github/Mohamed-Mohamed-Ibrahim/Code-Generation-and-Guarding/blob/SFT/sft/peft-sft-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
!pip install peft transformers[torch] trl bitsandbytes datasets wandb -U --q

In [ ]:
import os
import random, math
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

## Wandb

In [ ]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

In [ ]:
import wandb
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mohamed-mohamed-ibrahim (mohamed-mohamed-ibrahim-alexandria-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
os.environ["WANDB_PROJECT"] = "sft_nlp_lab"  # name your W&B project
os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # log all model checkpoints

In [ ]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
dataset

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

In [ ]:
shuffled_dataset = dataset.shuffle(seed=42)
split_ds_1 = shuffled_dataset.train_test_split(test_size=0.1)
split_ds_2 = split_ds_1['test'].train_test_split(test_size=0.5)

train_dataset = split_ds_1["train"].select(range(3000))
eval_dataset = split_ds_2["train"].select(range(2000))
test_dataset = split_ds_2["test"].select(range(10))

In [ ]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 49626
})

Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order Let's roll! The ball's in our court! ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_numbers)
# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```


In [ ]:
# HF hub ID
BASE_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
BASE_MODEL_NAME = "arnir0/Tiny-LLM"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

## Hyperparameters

In [ ]:
# Lab
lora_r = 16
lora_target_modules = "all-linear"
bs =  1
gradient_accumulation_steps = 4
epochs = 1
lr = 2e-4
optim = "paged_adamw_8bit"
max_length = 1024

# Others

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    # device_map={"": torch.cuda.current_device()},
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=16,
    target_modules=lora_target_modules,  # Llama-style
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=lr,
    num_train_epochs=epochs,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    save_strategy="epoch",
    bf16=True,
    optim=optim,

    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=max_length,
    packing=False,

    # Scheduler
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Compute loss on model answers only
    completion_only_loss=True,

    # Logging
    logging_steps=50,
    report_to="wandb", # enables logging to W&B 😎
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,                # SFTConfig object
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
50,6.192100
100,5.696000
150,5.464900
200,5.178100
250,5.105000
300,5.071500
350,5.096500
400,4.994700
450,4.938500
500,5.006000


wandb: Adding directory to artifact (qlora-peft-output/checkpoint-750)... Done. 0.0s


TrainOutput(global_step=750, training_loss=5.1523907877604165, metrics={'train_runtime': 111.4312, 'train_samples_per_second': 26.922, 'train_steps_per_second': 6.731, 'total_flos': 19323611396352.0, 'train_loss': 5.1523907877604165, 'epoch': 1.0})

In [ ]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

# 6. Save adapter weights

In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

Adapter saved to: ./qlora-peft-output/adapter


In [ ]:
!tree


.
├── merged-model
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── tokenizer.model
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── tokenizer.model
│   ├── checkpoint-3
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── tokenizer.model
│   │   ├── trainer_state.json
│   │   └── training_args.bin
│   ├── checkpoint-750
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "Qwen/Qwen2-1.5B-Instruct"
BASE_ID = "arnir0/Tiny-LLM"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...
Loading adapter onto base model...
Merging LoRA adapter into base model weights...

Merged model saved to: ./merged-model



In [ ]:
!tree

.
├── merged-model
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   └── tokenizer.model
├── qlora-peft-output
│   ├── adapter
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── README.md
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   └── tokenizer.model
│   ├── checkpoint-3
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth
│   │   ├── scheduler.pt
│   │   ├── special_tokens_map.json
│   │   ├── tokenizer_config.json
│   │   ├── tokenizer.json
│   │   ├── tokenizer.model
│   │   ├── trainer_state.json
│   │   └── training_args.bin
│   ├── checkpoint-750
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   ├── optimizer.pt
│   │   ├── README.md
│   │   ├── rng_state.pth

# Run merged-model inference

In [ ]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"Output: {p['output']}\n")
    output = generate(wrapped)
    print(output)
    print("\n", "-" * 120, "\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


==================== MERGED MODEL OUTPUT ====================

Output: ```python
def reverse_order(text):
  # Split the text into sentences
  sentences = text.split(".")
  # Reverse the order of the words in each sentences
  for index, sentence in enumerate(sentences):
    words = sentence.split(" ")
    reversed_words = words[::-1]
    sentences[index] = " ".join(reversed_words)
  # Join the sentences in the original order
  reversed_text = ". ".join(sentences)
  # Return the result
  return reversed_text

reverse_order("The quick brown fox jumped over the lazy dog.")

# Output: dog. lazy the over jumped fox brown quick The
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Create a Python script that reverses the order of the words in each sentence in a given text The quick brown fox jumped over the lazy dog.
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: using System.Collections;
using System.Collections.Generic;
using UnityEngine;
public class PlayerMovement : MonoBehaviour {
    public float moveSpeed = 5f;
    void Update() {
        Vector3 movement = new Vector3(Input.GetAxis("Horizontal"), 0f, Input.GetAxis("Vertical"));
        transform.position += movement * Time.deltaTime * moveSpeed;
    }
}



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Create a string matching algorithm in Python using Regular Expressions
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
import spacy
nlp = spacy.load('en_core_web_sm')
while True:
    message = input('Enter a chat message: ')
    doc = nlp(message)
    for token in doc:
        print(token.text, token.lemma_, token.pos_)
    if message == 'exit':
        break
# This will perform tokenization, lemmatization, and part-of-speech tagging on chat messages in real-time.
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: How to perform real-time chat analysis?
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
def calculate_sum(numbers):
    return sum(numbers)
```



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Optimize this python code to reduce the number of lines and make it more efficient def calculate_sum(numbers):
    sum = 0
    for n in numbers:
        sum += n
    return sum
Answer:









































































































































































































 ------------------------------------------------------------------------------------------------------------------------ 

Output: ```python
from sklearn.neighbors import KNeighborsClassifier

# Create the KNN classifier
knn = KNeighborsClassifier(n_neighbors = 3)

# Fit the classifier to the data
knn.fit(X, y)

# Predict the labels for the test set
y_pred = knn.predict(X_test)
```

You are a personalized assistant that knows details about the user based on prior fine-tuning data.

Question: Creat

In [ ]:
trainer.evaluate()

{'eval_loss': 4.940535068511963,
 'eval_runtime': 13.5336,
 'eval_samples_per_second': 147.781,
 'eval_steps_per_second': 18.473,
 'eval_entropy': 5.2256853332519535,
 'eval_num_tokens': 465078.0,
 'eval_mean_token_accuracy': 0.25044004517793655,
 'epoch': 1.0}

In [ ]:
metrics = trainer.evaluate()
ppl = math.exp(metrics['eval_loss']) if metrics['eval_loss'] < 20 else float('inf')
print('KL-SFT post-train eval_loss:', metrics['eval_loss'], 'ppl:', ppl)
print('Eval losses over steps:', [(h.get('step'), h.get('eval_loss')) for h in trainer.state.log_history if 'eval_loss' in h])


KL-SFT post-train eval_loss: 4.940535068511963 ppl: 139.845056231014
Eval losses over steps: [(750, 4.940535068511963), (750, 4.940535068511963)]


# Resources

- https://wandb.ai/capecape/alpaca_ft/reports/How-to-Fine-tune-an-LLM-Part-3-The-HuggingFace-Trainer--Vmlldzo1OTEyNjMy
- https://youtu.be/t1caDsMzWBk
- https://youtu.be/cayFaWkI39A